In [1]:
#import package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from pathlib import Path

In [2]:
notebook_dir = Path.cwd()
retail_dir = notebook_dir.parent
if not (retail_dir / "src" / "data_load_retail.py").exists():
    retail_dir = notebook_dir.parent.parent

os.chdir(retail_dir)
from src.data_load_retail import load_credit_data, apply_labels, CODE_LABELS


df = load_credit_data()
df = apply_labels(df)


In [5]:
from src.train_split import split_pd_data

splits = split_pd_data(df, target_col='target', test_size=0.25)
train_df, test_df = splits['train'], splits['test']

train_df.to_csv("train_data.csv", index=False)
test_df.to_csv("test_data.csv", index=False)

print(np.mean(train_df.target))

print(np.mean(test_df.target))


0.3
0.3


In [42]:
from src.IV_WOE import iv_summary

results_list = []


categorical_features = set(
    train_df.select_dtypes(include=["object", "str", "category", "bool"]).columns
)
categorical_features.update(
    col for col in train_df.columns if col.endswith("_code")
)

for feature in train_df.columns.drop("target"):
    # Prefer the coded version when both columns exist.
    if not feature.endswith("_code") and f"{feature}_code" in train_df.columns:
        continue

    categorical = [feature] if feature in categorical_features else []
    results = iv_summary(
        train_df,
        target="target",
        features=[feature],
        bins=5,
        categorical_features=categorical
    )
    results_list.append(results)

iv_results_df = pd.concat(results_list, ignore_index=True)
iv_results_df = iv_results_df.sort_values(
    "iv", ascending=False, na_position="last"
).reset_index(drop=True)

display(iv_results_df)
iv_results_df.to_csv("iv_results.csv", index=False)

,feature,iv,strength
0,checking_account_status_code,0.624314,suspicious (check leakage)
1,credit_history_code,0.259934,medium
2,savings_account_code,0.230510,medium
3,duration_months,0.190452,medium
4,property_code,0.150028,medium
5,purpose_code,0.144816,medium
6,employment_since_code,0.126575,medium
7,other_installment_plans_code,0.092840,weak
8,housing_code,0.090245,weak
9,credit_amount,0.070356,weak
